# kor-minish v3 fine-tune (Colab T4)

**v2 대비 추가**:
- ✅ 한글 vocab 30K 확실히 주입 (GitHub raw에서 받음)
- ✅ AI Hub 일상대화 데이터 추가:
  - Q&A 인접 turn 50K
  - sessionSummary paraphrase 30K
  - 같은 topic 묶음 10K 그룹
- ✅ PAWS-X는 paraphrase=1만 선별 (negative 비중 축소)

**예상**: T4 약 90~120분

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q 'sentence-transformers>=3.0' 'model2vec[distill]>=0.8' 'datasets>=2.19' 'huggingface_hub>=0.24'

In [ ]:
# v3 학습 데이터 다운로드 (GitHub raw)
BASE_URL = 'https://raw.githubusercontent.com/johunsang/kor-minish/main/data/v3'
!mkdir -p data
for fname in ['vocab_ko.txt', 'aihub_qa_sample.csv', 'aihub_summary_sample.csv', 'aihub_paraphrase_sample.jsonl']:
    !wget -q $BASE_URL/$fname -O data/$fname && ls -lh data/$fname

In [ ]:
from datasets import load_dataset
from sentence_transformers import InputExample
import csv, json, random
from itertools import combinations
random.seed(42)

NLI_SCORE = {0: 1.0, 1: 0.5, 2: 0.0}
train_examples = []
stats = {}

def add(name, count):
    stats[name] = count
    print(f'  + {name}: {count:,}')

# 1) KLUE-STS
ds = load_dataset('klue', 'sts', split='train')
n = 0
for row in ds:
    train_examples.append(InputExample(texts=[row['sentence1'], row['sentence2']], label=row['labels']['label']/5.0))
    n += 1
add('klue-sts', n)

# 2) KLUE-NLI (entailment/neutral/contradiction)
ds = load_dataset('klue', 'nli', split='train')
n = 0
for row in ds:
    train_examples.append(InputExample(texts=[row['premise'], row['hypothesis']], label=NLI_SCORE.get(row['label'], 0.5)))
    n += 1
add('klue-nli', n)

# 3) PAWS-X 한국어 — paraphrase=1만 선별 (positive 강화)
ds = load_dataset('paws-x', 'ko', split='train')
n = 0
for row in ds:
    if row['label'] == 1:
        train_examples.append(InputExample(texts=[row['sentence1'], row['sentence2']], label=1.0))
        n += 1
add('paws-x-ko (positive only)', n)

# 4) AI Hub Q&A (인접 turn, score=0.7 = 약한 positive: 관련 있지만 동의어는 아님)
with open('data/aihub_qa_sample.csv', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    n = 0
    for row in reader:
        q = row['question'].strip()
        a = row['answer'].strip()
        if q and a:
            train_examples.append(InputExample(texts=[q, a], label=0.7))
            n += 1
add('aihub-qa', n)

# 5) AI Hub Summary paraphrase (score=1.0)
with open('data/aihub_summary_sample.csv', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    n = 0
    for row in reader:
        train_examples.append(InputExample(texts=[row['sentence1'], row['sentence2']], label=float(row['score'])))
        n += 1
add('aihub-summary', n)

# 6) AI Hub Topic groups → paraphrase pair (그룹당 최대 3쌍)
n = 0
with open('data/aihub_paraphrase_sample.jsonl', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line.strip())
        examples = obj.get('examples', [])
        pairs = list(combinations(examples, 2))
        random.shuffle(pairs)
        for a, b in pairs[:3]:
            train_examples.append(InputExample(texts=[a, b], label=1.0))
            n += 1
add('aihub-topic-paraphrase', n)

random.shuffle(train_examples)
print(f'\n총 학습 쌍: {len(train_examples):,}')

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

BASE = 'BAAI/bge-m3'
EPOCHS = 1
BATCH = 16

model = SentenceTransformer(BASE, device='cuda')
loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH)
loss = losses.CosineSimilarityLoss(model)
warmup = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup,
    output_path='bge-m3-ko-finetuned-v3',
    show_progress_bar=True,
)
print('fine-tune 완료')

In [ ]:
# Re-distill with Korean vocab
from model2vec.distill import distill

vocab = [l.strip() for l in open('data/vocab_ko.txt', encoding='utf-8') if l.strip()]
print(f'vocab_ko: {len(vocab):,} tokens')

m2v = distill(
    model_name='bge-m3-ko-finetuned-v3',
    vocabulary=vocab,
    pca_dims=256,
    device='cuda',
)
m2v.save_pretrained('kor-minish-bge-m3-ko-v3')
print(f'saved → kor-minish-bge-m3-ko-v3 (vocab size: {m2v.embedding.shape[0]:,})')

In [ ]:
# v1 vs v3 비교
import numpy as np
from model2vec import StaticModel

v3 = StaticModel.from_pretrained('kor-minish-bge-m3-ko-v3')
v1 = StaticModel.from_pretrained('hysnnnn/kor-minish-bge-m3-ko')

def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

SYN = [
    ('월급이 인상되었다', '급여가 올랐다'),
    ('이 책은 정말 흥미롭다', '이 도서는 매우 재미있다'),
    ('서울에서 부산까지 KTX', '서울발 부산행 고속열차'),
    ('돈 돌려받고 싶어요', '환불 신청 방법'),
    ('비번 까먹었어요', '비밀번호 찾기'),
    ('아 그냥 안 살래요', '주문 취소'),
    ('할인 코드 어디에 넣어요', '쿠폰 사용법'),
    ('강아지가 공원에서 뛰어논다', '개가 공원에서 놀고 있다'),
]
UNREL = [
    ('김치찌개 레시피', '자동차 엔진 정비'),
    ('주식 시장 폭락', '강아지 산책 코스'),
]
import statistics as st
v1s, v3s, v1u, v3u = [], [], [], []
print(f'{"문장쌍":50s}  v1     v3     diff')
print('=== 동의 (높을수록 좋음) ===')
for a, b in SYN:
    s1, s3 = cos(*v1.encode([a, b])), cos(*v3.encode([a, b]))
    v1s.append(s1); v3s.append(s3)
    print(f'{a[:22]:22s} ↔ {b[:22]:22s}  {s1:+.3f}  {s3:+.3f}  {s3-s1:+.3f}')
print('=== 무관 ===')
for a, b in UNREL:
    s1, s3 = cos(*v1.encode([a, b])), cos(*v3.encode([a, b]))
    v1u.append(s1); v3u.append(s3)
    print(f'{a[:22]:22s} ↔ {b[:22]:22s}  {s1:+.3f}  {s3:+.3f}  {s3-s1:+.3f}')
print()
print(f'동의 평균: v1={st.mean(v1s):+.3f}  v3={st.mean(v3s):+.3f}')
print(f'무관 평균: v1={st.mean(v1u):+.3f}  v3={st.mean(v3u):+.3f}')
print(f'Margin:    v1={st.mean(v1s)-st.mean(v1u):+.3f}  v3={st.mean(v3s)-st.mean(v3u):+.3f}')

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('kor-minish-bge-m3-ko-v3', 'zip', 'kor-minish-bge-m3-ko-v3')
files.download('kor-minish-bge-m3-ko-v3.zip')